# Covarianza y correlación entre X₁ y X₂

- $X_1 \sim N(0,1)$
- $X_2 = -X_1$ si $|X_1|<1$, y $X_2 = X_1$ si $|X_1|>1$

**Teoría.** Como $X_2$ es $\pm X_1$ con una regla simétrica, $X_2 \sim N(0,1)$ también, así que $E[X_2]=0$ y $\mathrm{Var}(X_2)=1$. Entonces

$$\mathrm{Cov}(X_1,X_2)=E[X_1X_2]=-E[X_1^2\,\mathbb 1_{|X_1|<1}]+E[X_1^2\,\mathbb 1_{|X_1|>1}] = 1-2\,E[X_1^2\,\mathbb 1_{|X_1|<1}]$$

con $E[X_1^2\,\mathbb 1_{|X_1|<1}] = (2\Phi(1)-1)-2\varphi(1)\approx 0.1987$, por lo que $\mathrm{Cov}=\mathrm{Corr}\approx 0.6025$.

Son **dependientes** (X₂ es función de X₁) pero la correlación no es ni ±1 ni 0.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

rng = np.random.default_rng(42)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})

def make_x2(x1, c=1.0):
    return np.where(np.abs(x1) < c, -x1, x1)

# Covarianza teórica para umbral c
def cov_teorica(c=1.0):
    inner = (2*norm.cdf(c) - 1) - 2*c*norm.pdf(c)   # E[X^2 1{|X|<c}]
    return 1 - 2*inner

## 1. Simulación

In [ ]:
N = 200_000
x1 = rng.standard_normal(N)
x2 = make_x2(x1)

cov_emp  = np.cov(x1, x2)[0, 1]
corr_emp = np.corrcoef(x1, x2)[0, 1]
print(f"Cov  empírica = {cov_emp:.4f} | teórica = {cov_teorica():.4f}")
print(f"Corr empírica = {corr_emp:.4f} | teórica = {cov_teorica():.4f}")
print(f"media X2 = {x2.mean():.4f}, var X2 = {x2.var():.4f}  (esperado 0 y 1)")

## 2. Dispersión X₁ vs X₂ y distribución de X₂

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

idx = rng.choice(N, 4000, replace=False)
ax[0].scatter(x1[idx], x2[idx], s=5, alpha=.5)
ax[0].axvline(-1, color="k", ls="--", lw=.8); ax[0].axvline(1, color="k", ls="--", lw=.8)
ax[0].set(xlabel="$X_1$", ylabel="$X_2$",
          title=f"Dispersión (Cov={cov_emp:.3f}, Corr={corr_emp:.3f})")

ax[1].hist(x2, bins=100, density=True, alpha=.6, label="$X_2$ simulada")
g = np.linspace(-4, 4, 400)
ax[1].plot(g, norm.pdf(g), "r", label="N(0,1)")
ax[1].set(xlabel="$X_2$", title="X₂ también es N(0,1)"); ax[1].legend()
plt.tight_layout(); plt.show()

## 3. Convergencia de covarianza y correlación al crecer la muestra

In [ ]:
n = np.arange(1, N+1)
m1, m2 = np.cumsum(x1)/n, np.cumsum(x2)/n
cov_run = np.cumsum(x1*x2)/n - m1*m2
v1 = np.cumsum(x1**2)/n - m1**2
v2 = np.cumsum(x2**2)/n - m2**2
corr_run = cov_run / np.sqrt(v1*v2)

start = 50
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(n[start:], cov_run[start:],  label="Covarianza")
ax.plot(n[start:], corr_run[start:], label="Correlación", ls="--")
ax.axhline(cov_teorica(), color="r", lw=1, label=f"Teórico = {cov_teorica():.4f}")
ax.set(xscale="log", xlabel="n (tamaño de muestra)", title="Convergencia de los estimadores")
ax.legend(); plt.show()

## 4. Variabilidad del estimador (muchas réplicas)

In [ ]:
R, n_rep = 2000, 500
covs, corrs = np.empty(R), np.empty(R)
for i in range(R):
    a = rng.standard_normal(n_rep); b = make_x2(a)
    covs[i]  = np.cov(a, b)[0, 1]
    corrs[i] = np.corrcoef(a, b)[0, 1]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a_, d, t in zip(ax, (covs, corrs), ("Covarianza", "Correlación")):
    a_.hist(d, bins=40, alpha=.7)
    a_.axvline(cov_teorica(), color="r", label="teórico")
    a_.axvline(d.mean(), color="k", ls="--", label=f"media={d.mean():.3f}")
    a_.set(title=f"{t} en {R} réplicas de n={n_rep}"); a_.legend()
plt.tight_layout(); plt.show()

## 5. Extra: ¿qué pasa si cambiamos el umbral $c$ (en vez de 1)?

Con $c=0$ se tiene $X_2=X_1$ (corr = 1); con $c\to\infty$, $X_2=-X_1$ (corr = −1).
La correlación se anula en $c\approx1.54$ aunque $X_2$ sigue siendo función de $X_1$.

In [ ]:
cgrid = np.linspace(0, 3.5, 36)
emp = []
for c in cgrid:
    a = rng.standard_normal(100_000); b = make_x2(a, c)
    emp.append(np.corrcoef(a, b)[0, 1])

cf = np.linspace(0, 3.5, 400)
c0 = cf[np.argmin(np.abs(cov_teorica(cf)))]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(cf, cov_teorica(cf), "r", label="Teórica (Cov = Corr)")
ax.plot(cgrid, emp, "o", ms=4, label="Simulada")
ax.axhline(0, color="k", lw=.8); ax.axvline(1, color="gray", ls=":", label="c = 1 (enunciado)")
ax.axvline(c0, color="green", ls=":", label=f"corr = 0 en c≈{c0:.2f}")
ax.set(xlabel="umbral c", ylabel="correlación", title="Correlación en función del umbral"); ax.legend()
plt.show()